In [12]:
# https://colab.research.google.com/drive/1JMLa53HDuA-i7ZBmqV7ZnA3c_fvtXnx-?usp=sharing#scrollTo=wJpXpmjEYC_T
# https://www.bilibili.com/video/BV1BbFaeVE4W  PyTorch手搓Transformer
# https://github.com/hankinghu/literature-books/tree/master

In [1]:

import torch
import torch.nn as nn
from torch.nn import functional as F
import textwrap
import random

# 超参数
file_name="sanguo-all.txt"
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
wrap_width = 50
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.1
# ------------

torch.manual_seed(1337)


In [2]:

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open(file_name, 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y


In [11]:
# 傻瓜模型
class LanguageModel(nn.Module):
    def __init__ (self):
        super().__init__ ()

    def forward(self,idx,targets=None):
        B,T=idx.shape #B= batch size,T= block size，数据为token(整数)形式
        random_tensor = torch.rand(B,T,vocab_size,device=device) #
        logits = random_tensor /random_tensor.sum(dim=-1, keepdim=True)
        loss = None
        return logits, loss
    
    def generate(self, token_sequ, max_new_tokens):
        # token_sequ已知的上文,max_new_tokens是续写的长度(B，T)
        for _ in range(max_new_tokens):
            tokens_input = token_sequ[:, -block_size: ]
            logits, loss = self.forward(tokens_input)  # logits,(B, T, vocab size)
            logits = logits[:,-1,:] #只取字符串最后一个,(概率分布向量格式)
            probs =F.softmax(logits,dim=-1)
            token_next = torch.multinomial(probs,num_samples=1)# 概率分布向量-->one-hot 向量-->整数token
            token_sequ =torch.cat((token_sequ, token_next), dim=1)
        new_tokens =token_sequ[:,-max_new_tokens:]
        return new_tokens

In [17]:
model= LanguageModel()
model.to(device)

max_new_tokens =100
start_idx = random.randint(0, len(val_data)-block_size-max_new_tokens)
#上文内容
context = torch.zeros((1, block_size), dtype=torch.long, device=device)# (B, T)B = 1,T = block size
context[0,:]=val_data[start_idx:start_idx+block_size]
context_str =decode(context[0].tolist())#一阶张量
wrapped_context_str = textwrap.fill(context_str, width=wrap_width)
#真实下文
real_next_tokens = torch.zeros((1,max_new_tokens), dtype=torch.long, device=device)
real_next_tokens[0, :]= val_data[start_idx+block_size: start_idx+block_size+max_new_tokens]
real_next_tokens_str = decode(real_next_tokens[0].tolist())# 一阶张量
wrapped_real_next_tokens_str = textwrap.fill(real_next_tokens_str, width=wrap_width)
#生成下文
generated_tokens = model.generate(context, max_new_tokens)
generated_str =decode(generated_tokens[0].tolist())
wrapped_generated_str = textwrap.fill(generated_str, width=wrap_width)

print("上文内容:")
print(wrapped_context_str)
print("生成内容:")
print(wrapped_generated_str)
print("真实上文内容:")
print(wrapped_real_next_tokens_str)

上文内容:
大惊曰：“中邓艾之计矣！”遂传令教夏侯霸、张翼各弃狄道而退。于是
生成内容:
滚梦淫艰仆旙颌毙截壤梳襄鬻欷屏陷耻羽吉晤蔡莽涣犊凰弱丽低沟瘁隔讲徙举影互捎鹏摧囚卓谢渠颖耀宝贺厘恰循
勃七楙淇柄焕馆槛降倦服谷紞话蠹叠察崎薨法夫最香垒出贝困咬涝遥携叔摧肤驸.商蕤刎鲜混善屈项杯骑龟伏缢癣
真实上文内容:
蜀兵皆退于汉中。维自断后，只听得背后鼓声不绝，维退入剑阁之时，方知火鼓二十余处，皆虚设也。维收兵退屯
于钟提。 且说后主因姜维有洮西之功，降诏封维为大将军。维受了职，上表谢恩毕，再议出师伐魏之策。正是：
